# Dataset 1
Cleaning and Overview

In [69]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sys
import os

# Set notebook_dir to the current working directory
notebook_dir = os.getcwd()

# append scripts directory to path to import utils functions
scripts_dir = os.path.join(notebook_dir, '../scripts')
sys.path.append(scripts_dir)
from utils import get_timestamp


In [70]:

# Load delete_datasets
df = pd.read_csv('../data/raw/ds_1_raw_womens-educational-attainment-vs-fertility_2025-07-18.csv')


print("Head:")
print(df.head())
print("Shape:", df.shape)
print("Info:")
print(df.info())
print("Describe:")
print(df.describe())
print("Number of duplicates:", df.duplicated().sum())
if df.duplicated().any():
    df.drop_duplicates(inplace=True)
    print("Duplicates dropped. New shape:", df.shape)
print("Missing values per column:")
print(df.isnull().sum())

Head:
        Entity Code  Year  \
0  Afghanistan  AFG  1950   
1  Afghanistan  AFG  1951   
2  Afghanistan  AFG  1952   
3  Afghanistan  AFG  1953   
4  Afghanistan  AFG  1954   

   Fertility rate - Sex: all - Age: all - Variant: estimates  \
0                                              7.248           
1                                              7.260           
2                                              7.260           
3                                              7.266           
4                                              7.254           

   Combined - average years of education for 15-64 years female youth and adults  \
0                                               0.08                               
1                                                NaN                               
2                                                NaN                               
3                                                NaN                               
4             

In [71]:
# Filter only years from 1950 to 2020
df = df[(df['Year'] >= 1950) & (df['Year'] <= 2020)]
# print the shape after filtering
print("Shape after filtering years 1950-2020:", df.shape)

Shape after filtering years 1950-2020: (18943, 7)


In [72]:
# Make region Mapping

df_region = pd.read_csv('../data/raw/ds_1_raw_womens-educational-attainment-vs-fertility_2025-07-18.csv')

print("Head:")
print(df.head())
print("Shape:", df.shape)
print("Info:")
print(df.info())
print("Describe:")
print(df.describe())
print("Number of duplicates:", df.duplicated().sum())
if df.duplicated().any():
    df.drop_duplicates(inplace=True)
    print("Duplicates dropped. New shape:", df.shape)
print("Missing values per column:")
print(df.isnull().sum())

# Build country-to-region mapping using only relevant columns
region_df = df_region[['Entity', 'World regions according to OWID']].dropna(subset=['World regions according to OWID'])
region_map = region_df.drop_duplicates(subset=['Entity']).set_index('Entity')['World regions according to OWID'].to_dict()

# print region mapping
print("Region mapping:")
print(region_map)

# for entity "World" assign "World" as region
region_map['World'] = 'World'

# Fill out the 'Region' column in the original DataFrame
df['World regions according to OWID'] = df['Entity'].map(region_map)

# Check if all regions are mapped
if df['World regions according to OWID'].isnull().any():
    print("Warning: Some regions are not mapped. Check the mapping.")
else:
    print("All regions are successfully mapped.")
    
# Print which regions aren't mapped
unmapped_regions = df[df['World regions according to OWID'].isnull()]['Entity'].unique()
if len(unmapped_regions) > 0:
    print("Unmapped regions:", unmapped_regions)
else:
    print("No unmapped regions found.")
    
#Delete those unmapped regions
df = df[~df['Entity'].isin(unmapped_regions)]
print("Shape after removing unmapped regions:", df.shape)

Head:
        Entity Code  Year  \
0  Afghanistan  AFG  1950   
1  Afghanistan  AFG  1951   
2  Afghanistan  AFG  1952   
3  Afghanistan  AFG  1953   
4  Afghanistan  AFG  1954   

   Fertility rate - Sex: all - Age: all - Variant: estimates  \
0                                              7.248           
1                                              7.260           
2                                              7.260           
3                                              7.266           
4                                              7.254           

   Combined - average years of education for 15-64 years female youth and adults  \
0                                               0.08                               
1                                                NaN                               
2                                                NaN                               
3                                                NaN                               
4             

In [73]:
# save file using naming convention <type>_<id>_<state>_<descriptive name>[_version]_<timestamp>.<file_extension>
timestamp = get_timestamp()
df.to_csv(f'../data/processed/ds_1_filtered_womans-educational-attainment-vs-fertility_{timestamp}.csv')

In [74]:
# write metadata as a dictionary, not a list
metadata = {
    "chart": {
        "title": "Women's educational attainment vs. fertility rate (Filtered, 1950-2020, mapped regions)",
        "subtitle": "Filtered to years 1950-2020, duplicates removed, and only countries with mapped OWID regions included. The dataset summarizes the total number of births a woman would have and the average years of formal education received by women between the ages of 15 and 64.",
        "citation": "UN, World Population Prospects (2024); Barro and Lee (2015); Lee and Lee (2016)",
        "originalChartUrl": "https://ourworldindata.org/grapher/womens-educational-attainment-vs-fertility?v=1&csvType=full&useColumnShortNames=false",
        "selection": []
    },
    "columns": {
        "Entity": {
            "titleShort": "Country or region",
            "descriptionShort": "Country or region name as defined by OWID."
        },
        "Year": {
            "titleShort": "Year",
            "descriptionShort": "Calendar year of observation."
        },
        "Fertility rate - Sex: all - Age: all - Variant: estimates": {
            "titleShort": "Fertility rate, total",
            "titleLong": "Fertility rate, total - UN WPP",
            "descriptionShort": "The average number of live births a hypothetical cohort of women would have at the end of their reproductive period if they were subject during their whole lives to the fertility rates of a given period and if they were not subject to mortality.",
            "unit": "live births per woman",
            "timespan": "1950-2020",
            "type": "Numeric",
            "owidVariableId": 950920
        },
        "Combined - average years of education for 15-64 years female youth and adults": {
            "titleShort": "Average years of schooling (women aged 15–64)",
            "titleLong": "Average years of schooling (women aged 15–64)",
            "descriptionShort": "Average years of formal education for women aged 15-64.",
            "unit": "years",
            "timespan": "1950-2020",
            "type": "Numeric",
            "owidVariableId": 809140
        },
        "Population (historical)": {
            "titleShort": "Population (historical)",
            "descriptionShort": "Population by country, available from 1950 to 2020.",
            "unit": "people",
            "timespan": "1950-2020",
            "type": "Integer",
            "owidVariableId": 953903
        },
        "World regions according to OWID": {
            "titleShort": "World regions according to OWID",
            "descriptionShort": "World region as mapped by OWID.",
            "type": "Continent",
            "owidVariableId": 900801
        }
    },
    "filtered": {
        "file": f"../data/processed/ds_1_filtered_womans-educational-attainment-vs-fertility_{timestamp}.csv",
        "shape": [df.shape[0], df.shape[1]],
        "columns": [
            "Entity",
            "Year",
            "Fertility rate - Sex: all - Age: all - Variant: estimates",
            "Combined - average years of education for 15-64 years female youth and adults",
            "Population (historical)",
            "World regions according to OWID"
        ],
        "filtersApplied": [
            "Years filtered to 1950-2020",
            "Duplicates removed",
            "Rows with unmapped regions removed"
        ],
        "processingSteps": [
            "Loaded raw data",
            "Dropped duplicate rows",
            "Filtered for years 1950-2020",
            "Mapped countries to OWID regions",
            "Removed rows with unmapped regions"
        ],
        "dateFiltered": timestamp
    },
    "dateDownloaded": "2025-07-18",
    "activeFilters": {
        "years": [1950, 2020],
        "regions": "Mapped to OWID regions"
    }
}

metadata_file = f'../data/processed/md_1_filtered_womans-educational-attainment-vs-fertility_{timestamp}.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=4)